In [1]:
from ibm_watsonx_ai import APIClient
from ibm_watsonx_ai import Credentials

In [2]:
# Instantiate a client using credentials
credentials = Credentials(
      api_key = "******************",
      url = "https://us-south.ml.cloud.ibm.com"
)

client = APIClient(credentials)

In [ ]:
import os
import types
import pandas as pd
from botocore.client import Config
import ibm_boto3

def __iter__(self): return 0

# IBM Cloud Object Storage Credentials
cos_client = ibm_boto3.client(service_name='s3',
    ibm_api_key_id='9eC00Ve9AnZjI2e0ZzTnJ3K8uT8EBG8U-FZ1o2ofm8el',
    ibm_auth_endpoint="https://iam.cloud.ibm.com/identity/token",
    config=Config(signature_version='oauth'),
    endpoint_url='https://s3.direct.us-south.cloud-object-storage.appdomain.cloud')

bucket = 'deloitteinventoryandforecast-donotdelete-pr-fuxzt5avxe1dgg'

# List all CSV files in the bucket
csv_files = [obj['Key'] for obj in cos_client.list_objects_v2(Bucket=bucket)['Contents'] if obj['Key'].endswith('.csv')]

# Dictionary to store all DataFrames
dataframes = {}

# Loop through each CSV and read it into a DataFrame dynamically
for file in csv_files:
    try:
        # Get the file object
        obj = cos_client.get_object(Bucket=bucket, Key=file)
        body = obj['Body']
        
        # Check if the file is empty
        if obj['ContentLength'] == 0:
            print(f"Warning: The file {file} is empty and will be skipped.")
            continue
        
        # Ensure the body object is iterable
        if not hasattr(body, "__iter__"): 
            body.__iter__ = types.MethodType(__iter__, body)
        
        # Create a lowercase DataFrame name with a consistent format
        df_key = os.path.splitext(file)[0].lower()  # Remove file extension and convert to lowercase
        df_name = f"{df_key}_df"  # Example: 'DC.csv' → 'dc_df'
        
        # Read CSV into DataFrame and store it in dictionary
        dataframes[df_name] = pd.read_csv(body)
        
        print(f"Successfully read {file} into DataFrame {df_name}")
    
    except pd.errors.EmptyDataError:
        print(f"Warning: The file {file} is empty or contains no columns and will be skipped.")
    except Exception as e:
        print(f"Error reading {file}: {e}")

# Print all DataFrame keys to verify
print("DataFrames Created:", dataframes.keys())

In [4]:
dc = dataframes['dc_df']
plant = dataframes['plant_df']
retailer = dataframes['retailer_df']
product = dataframes['product_df']
time_period = dataframes['time_period_df']
dc_product = dataframes['dc_product_df']
dc_retailer = dataframes['dc_retailer_df']
plant_dc = dataframes['plant_dc_df']
version = dataframes['version_df']
plant_product = dataframes['plant_product_df']
parameter=dataframes['parameter_df']
retailer_product=dataframes['retailer_product_df']




In [ ]:
# Assume version_to_solve is given
version_to_solve = "V1"
filtered_version = version[version["VERSION_ID"] == version_to_solve]
print(filtered_version)


In [ ]:
# Step 1: Filter the version table for the given version_to_solve
version_row = version[version['VERSION_ID'] == version_to_solve].iloc[0]

# Step 2: Extract start and end periods
start_year = version_row['START_YEAR']  # 2025
start_period = version_row['START_PERIOD_IN_YEAR']  # 1
end_year = version_row['END_YEAR']  # 2025
end_period = version_row['END_PERIOD_IN_YEAR']  # 3

# Step 3: Filter time_period based on extracted values
filtered_time_period = time_period[
    ((time_period['YEAR'] == start_year) & (time_period['PERIOD_IN_YEAR'] >= start_period)) |
    ((time_period['YEAR'] == end_year) & (time_period['PERIOD_IN_YEAR'] <= end_period))
]

# Extract only the TIME_PERIOD_ID column
filtered_time_ids = filtered_time_period['TIME_PERIOD_ID'].tolist()

# Print to verify
print("Filtered Time Periods:", filtered_time_ids)


In [ ]:
print(filtered_time_period)

## DC-Costing: Flow of products 𝑘 between locations 𝑖 and 𝑗 in time 𝑡

In [8]:
import pandas as pd
from docplex.mp.model import Model


# Assuming your dataframes are:
# dc, plant, retailer, product, time_period, dc_product, dc_retailer, plant_dc, version


filtered_plant_dc = plant_dc[plant_dc['VERSION_ID'] == version_to_solve][['PLANT_ID', 'DC_ID']]
filtered_dc_retailer = dc_retailer[dc_retailer['VERSION_ID'] == version_to_solve][['DC_ID', 'RETAILER_ID']]

# Step 2: Create a unified origin-destination DataFrame
valid_od_df = pd.concat([
    filtered_plant_dc.rename(columns={'PLANT_ID': 'origin', 'DC_ID': 'destination'}),
    filtered_dc_retailer.rename(columns={'DC_ID': 'origin', 'RETAILER_ID': 'destination'})
], ignore_index=True).drop_duplicates().reset_index(drop=True)






In [ ]:
print(filtered_dc_retailer)

In [ ]:
print("Valid Origin-Destination Pairs:", valid_od_df.shape[0])
print(valid_od_df)  # Preview first few rows


In [11]:
mdl = Model(name="SupplyChainOptimization")


### Define Decision Variables Xijkt (Flow quantity from i to j for product

In [12]:
x_vars = mdl.integer_var_dict(
    ((i, j, k, t)
     for _, row in valid_od_df.iterrows()  # Iterate over valid (i, j) pairs
     for i, j in [(row['origin'], row['destination'])]  # Extract origin & destination
     for k in product['PRODUCT_ID'].unique()  # Ensure only 2 unique products
     for t in filtered_time_period['TIME_PERIOD_ID'].unique()),  # Use filtered time periods
    lb=0,
    name=lambda idx: f"X_{idx[0]}_{idx[1]}_{idx[2]}_t{idx[3]}"
)


In [ ]:
x_vars

## Retailer Costing: Storage of products 𝑘 at retailer 𝑗 in time 𝑡:


In [ ]:
plant_cost_cap = plant[plant['VERSION_ID'] == version_to_solve][['PLANT_ID', 'HOLDING_COST', 'STORAGE_CAPACITY']]
dc_cost_cap = dc[dc['VERSION_ID'] == version_to_solve][['DC_ID', 'HOLDING_COST', 'STORAGE_CAPACITY']]
retailer_cost_cap = retailer[retailer['VERSION_ID'] == version_to_solve][['RETAILER_ID', 'HOLDING_COST', 'STORAGE_CAPACITY']]

# Step 2: Create a unified location DataFrame (loc_df)
all_loc_cost_cap = pd.concat([
    plant_cost_cap.rename(columns={'PLANT_ID': 'location'}),
    dc_cost_cap.rename(columns={'DC_ID': 'location'}),
    retailer_cost_cap.rename(columns={'RETAILER_ID': 'location'})
], ignore_index=True).drop_duplicates().reset_index(drop=True)
all_loc_cost_cap

In [15]:
%%capture

# Step 1: Filter locations based on VERSION_ID
filtered_plants = plant[plant['VERSION_ID'] == version_to_solve][['PLANT_ID']]
filtered_dcs = dc[dc['VERSION_ID'] == version_to_solve][['DC_ID']]
filtered_retailers = retailer[retailer['VERSION_ID'] == version_to_solve][['RETAILER_ID', 'HOLDING_COST']]

# Step 2: Create a unified location DataFrame (loc_df)
loc_df = pd.concat([
    filtered_plants.rename(columns={'PLANT_ID': 'location'}),
    filtered_dcs.rename(columns={'DC_ID': 'location'}),
    filtered_retailers.rename(columns={'RETAILER_ID': 'location'})
], ignore_index=True).drop_duplicates().reset_index(drop=True)

# Step 3: Merge HOLDING_COST for retailers only
loc_df = loc_df.merge(filtered_retailers, how='left', left_on='location', right_on='RETAILER_ID').drop(columns=['RETAILER_ID'])

# **Fix: Ensure only one HOLDING_COST column exists**
if 'HOLDING_COST_x' in loc_df.columns and 'HOLDING_COST_y' in loc_df.columns:
    loc_df['HOLDING_COST'] = loc_df['HOLDING_COST_y']  # Use HOLDING_COST_y
    loc_df.drop(columns=['HOLDING_COST_x', 'HOLDING_COST_y'], inplace=True)
elif 'HOLDING_COST_x' in loc_df.columns:
    loc_df.rename(columns={'HOLDING_COST_x': 'HOLDING_COST'}, inplace=True)
elif 'HOLDING_COST_y' in loc_df.columns:
    loc_df.rename(columns={'HOLDING_COST_y': 'HOLDING_COST'}, inplace=True)

# Step 4: Keep only retailer locations for storage
retailer_locs = loc_df.dropna(subset=['HOLDING_COST']).reset_index(drop=True)  # Filter only retailers

# Step 5: Define Decision Variables Yjkt (Storage quantity at retailer j for product k at time t)
y_vars = mdl.integer_var_dict(
    ((j, k, t)
     for j in all_loc_cost_cap['location']
     for k in product['PRODUCT_ID']
     for t in filtered_time_period['TIME_PERIOD_ID']),  # Use filtered time periods dynamically
    lb=0,
    name=lambda idx: f"Y_{idx[0]}_{idx[1]}_t{idx[2]}"
)

In [ ]:
y_vars

 ## Plant Costing: Production of products 𝑘 at plant 𝑖 in time 𝑡:

In [17]:
%%capture
# Step 1: Ensure unique plants
filtered_plants = filtered_plants[['PLANT_ID']].drop_duplicates()

# Step 2: Ensure unique products
unique_products = product[['PRODUCT_ID']].drop_duplicates()

# Step 3: Define Decision Variables Pikt (production of product k at plant i in time period t)
p_vars = mdl.integer_var_dict(
    ((i, k, t) 
     for i in filtered_plants['PLANT_ID']  # Unique plants
     for k in unique_products['PRODUCT_ID']  # Unique products
     for t in filtered_time_period['TIME_PERIOD_ID']),  # Use filtered time periods dynamically
    lb=0,
    name=lambda idx: f"P_{idx[0]}_{idx[1]}_t{idx[2]}"
)

In [ ]:
p_vars

## Objective Function

In [19]:
# Production Cost (Product) - Pull unit cost from the product table
prod_cost = {}
productids = product['PRODUCT_ID'].unique()
for p in productids:
    try:
        prod_cost[p] = product.loc[product['PRODUCT_ID'] == p, 'UNIT_COST'].values[0] # Unit cost from the product table
    except:
        prod_cost[p] = 0



In [20]:
plantids = list(plant['PLANT_ID'].unique())
dcids = list(dc['DC_ID'].unique())
retailids = list(retailer['RETAILER_ID'].unique())


#Storage Cost (Location x Product)
storage_cost = {}
for location in plantids+dcids+retailids :
    for p in productids:
        try:
            if location in plantids:
                storage_cost[(location, p)] = plant.loc[plant['PLANT_ID'] == location, 'HOLDING_COST'].values[0] #Holding cost from the Plant table
            elif location in dcids:
                storage_cost[(location, p)] = dc.loc[dc['DC_ID'] == location, 'HOLDING_COST'].values[0] #Holding cost from the DC table
            elif location in retailids:
                storage_cost[(location, p)] = retailer.loc[retailer['RETAILER_ID'] == location, 'HOLDING_COST'].values[0] #Holding cost from the Retailer table
        except:
            storage_cost[(location, p)] = 0 #Cost 0 for the errors
        

In [21]:
# flow_cost_dict = {}

# # Add plant → DC flow costs
# for _, row in plant_dc.iterrows():
#     if row['VERSION_ID'] == version_to_solve:
#         i, j, cost = row['PLANT_ID'], row['DC_ID'], row['TRANSPORT_COST']
#         for k in product['PRODUCT_ID']:
#             for t in filtered_time_period['TIME_PERIOD_ID']:
#                 flow_cost_dict[(i, j, k, t)] = cost  # Assign transport cost

# # Add DC → Retailer flow costs
# for _, row in dc_retailer.iterrows():
#     if row['VERSION_ID'] == version_to_solve:
#         j, r, cost = row['DC_ID'], row['RETAILER_ID'], row['TRANSPORT_COST']
#         for k in product['PRODUCT_ID']:
#             for t in filtered_time_period['TIME_PERIOD_ID']:
#                 flow_cost_dict[(j, r, k, t)] = cost  # Assign transport cost


#### mass balance plant constraint

In [22]:
# Get the first time period (e.g., T37) dynamically
first_time_period = filtered_time_period['TIME_PERIOD_ID'].min()

# Create a mapping from TIME_PERIOD_ID to numeric index
time_index_mapping = {time_id: idx for idx, time_id in enumerate(filtered_time_period['TIME_PERIOD_ID'])}

# Create valid_flows as a set of valid (PLANT_ID, DC_ID) pairs
valid_flows = set(
    plant_dc.loc[plant_dc['VERSION_ID'] == version_to_solve, ['PLANT_ID', 'DC_ID']]
    .itertuples(index=False, name=None)  # Converts to tuples (PLANT_ID, DC_ID)
)

for i in plant['PLANT_ID']:  # Iterate over plants
    for k in product['PRODUCT_ID']:  # Iterate over products
        for t in filtered_time_period['TIME_PERIOD_ID']:  # Iterate over time periods
            
            # ✅ Get initial stock for the first time period
            initial_stock_value = plant_product.loc[
                (plant_product['PLANT_ID'] == i) & 
                (plant_product['PRODUCT_ID'] == k) & 
                (plant_product['VERSION_ID'] == version_to_solve),  
                'INITIAL_STOCK'
            ].values

            initial_stock_value = initial_stock_value[0] if initial_stock_value.size == 1 else 0

            # ✅ First time period (T37) uses initial stock
            if t == first_time_period:
                mdl.add_constraint(
                    initial_stock_value + p_vars[i, k, t] == 
                    y_vars[i, k, t] + mdl.sum(x_vars[i, j, k, t] for j in dc if (i, j) in valid_flows),
                    ctname=f"PlantMassBalance_{i}_{k}_t{t}"
                )
            else:
                previous_t = list(time_index_mapping.keys())[list(time_index_mapping.values()).index(time_index_mapping[t] - 1)]

                # ✅ Fix: Add Outflow (`X`) to Balance Equation
                mdl.add_constraint(
                    y_vars[i, k, previous_t] + p_vars[i, k, t] == 
                    y_vars[i, k, t] + mdl.sum(x_vars[i, j, k, t] for j in dc if (i, j) in valid_flows),
                    ctname=f"PlantMassBalance_{i}_{k}_t{t}"
                )


#### mass balance dc constraint

In [23]:

valid_inflows = set(
    plant_dc.loc[plant_dc['VERSION_ID'] == version_to_solve, ['PLANT_ID', 'DC_ID']]
    .drop_duplicates()
    .itertuples(index=False, name=None)
)



valid_outflows = set(
    dc_retailer.loc[dc_retailer['VERSION_ID'] == version_to_solve, ['DC_ID', 'RETAILER_ID']]
    .drop_duplicates()
    .itertuples(index=False, name=None)
)



In [24]:
for j in dc['DC_ID']:  # Iterate over DCs
    for k in product['PRODUCT_ID']:  # Iterate over products
        for t in filtered_time_period['TIME_PERIOD_ID']:  # Iterate over time periods
            
            # Get total initial stock for the specific version and DC
            initial_stock_value = dc_product.loc[
                (dc_product['DC_ID'] == j) & 
                (dc_product['PRODUCT_ID'] == k) & 
                (dc_product['VERSION_ID'] == version_to_solve),
                'INITIAL_STOCK'
            ].sum()  # SUM instead of raising an error

            # Compute inflow sum (ensures unique i values)
            inflow_sum = mdl.sum(
                x_vars[i, j, k, t] for i in set(plant_dc.loc[plant_dc['DC_ID'] == j, 'PLANT_ID'])
                if (i, j) in valid_inflows
            )

            # Compute outflow sum (ensures unique r values)
            outflow_sum = mdl.sum(
                x_vars[j, r, k, t] for r in set(dc_retailer.loc[dc_retailer['DC_ID'] == j, 'RETAILER_ID'])
                if (j, r) in valid_outflows
            )

            if t == first_time_period:  # If time period is T25
                mdl.add_constraint(
                    initial_stock_value + inflow_sum == y_vars[j, k, t] + outflow_sum,
                    ctname=f"DCMassBalance_{j}_{k}_t{t}"
                )
            else:  # For all other time periods
                previous_t = list(time_index_mapping.keys())[list(time_index_mapping.values()).index(time_index_mapping[t] - 1)]

                mdl.add_constraint(
                    y_vars[j, k, previous_t] + inflow_sum == y_vars[j, k, t] + outflow_sum,
                    ctname=f"DCMassBalance_{j}_{k}_t{t}"
                )


In [ ]:
print(plant_dc[plant_dc['VERSION_ID'] == version_to_solve])

### mass balance retailer constraint

In [ ]:

import os, types
import pandas as pd
from botocore.client import Config
import ibm_boto3

def __iter__(self): return 0

# @hidden_cell
# The following code accesses a file in your IBM Cloud Object Storage. It includes your credentials.
# You might want to remove those credentials before you share the notebook.

cos_client = ibm_boto3.client(service_name='s3',
    ibm_api_key_id='**********************',
    ibm_auth_endpoint="https://iam.cloud.ibm.com/identity/token",
    config=Config(signature_version='oauth'),
    endpoint_url='https://s3.direct.us-south.cloud-object-storage.appdomain.cloud')

bucket = 'deloitteinventoryandforecast-donotdelete-pr-fuxzt5avxe1dgg'
object_key = 'forcastdatav1.csv'

body = cos_client.get_object(Bucket=bucket,Key=object_key)['Body']
# add missing __iter__ method, so pandas accepts body as file-like object
if not hasattr(body, "__iter__"): body.__iter__ = types.MethodType( __iter__, body )

forecastdata = pd.read_csv(body)
forecastdata.head(10)


In [27]:
# import os, types
# import pandas as pd
# from botocore.client import Config
# import ibm_boto3

# def __iter__(self): return 0

# # @hidden_cell
# # The following code accesses a file in your IBM Cloud Object Storage. It includes your credentials.
# # You might want to remove those credentials before you share the notebook.

# cos_client = ibm_boto3.client(service_name='s3',
#     ibm_api_key_id='9eC00Ve9AnZjI2e0ZzTnJ3K8uT8EBG8U-FZ1o2ofm8el',
#     ibm_auth_endpoint="https://iam.cloud.ibm.com/identity/token",
#     config=Config(signature_version='oauth'),
#     endpoint_url='https://s3.direct.us-south.cloud-object-storage.appdomain.cloud')

# bucket = 'deloitteinventoryandforecast-donotdelete-pr-fuxzt5avxe1dgg'
# object_key = 'outputts_forecast.csv'

# body = cos_client.get_object(Bucket=bucket,Key=object_key)['Body']
# # add missing __iter__ method, so pandas accepts body as file-like object
# if not hasattr(body, "__iter__"): body.__iter__ = types.MethodType( __iter__, body )

# forecastdata = pd.read_csv(body)
# forecastdata.head(10)


In [ ]:
print( filtered_time_period['TIME_PERIOD_ID'].min())

In [29]:
# Get valid flows for DC to retailer
# valid_retailer_flows = set(
#     dc_retailer.loc[dc_retailer['VERSION_ID'] == version_to_solve, ['DC_ID', 'RETAILER_ID']]
#     .itertuples(index=False, name=None)
# )

valid_retailer_flows = set(
    dc_retailer.loc[dc_retailer['VERSION_ID'] == version_to_solve, ['DC_ID', 'RETAILER_ID']]
    .drop_duplicates()
    .itertuples(index=False, name=None)
)


# Get the first time period (T37)
first_time_period = filtered_time_period['TIME_PERIOD_ID'].min()

# Create a mapping from TIME_PERIOD_ID to numeric index
time_index_mapping = {time_id: idx for idx, time_id in enumerate(filtered_time_period['TIME_PERIOD_ID'])}

for r in retailer['RETAILER_ID']:  # Iterate over retailers
    for k in product['PRODUCT_ID']:  # Iterate over products
        for t in filtered_time_period['TIME_PERIOD_ID']:  # Iterate over time periods
            
            # Get initial stock for the specific version and first time period
            initial_stock = retailer_product.loc[
                (retailer_product['RETAILER_ID'] == r) & 
                (retailer_product['PRODUCT_ID'] == k) & 
                (retailer_product['VERSION_ID'] == version_to_solve),  # Ensure correct version
                'INITIAL_STOCK'
            ].values  # Returns an array
            
            # Assign initial stock value safely
            initial_stock_value = initial_stock[0] if initial_stock.size == 1 else 0

            # Get predicted demand from forecastdata
            predicted_demand = forecastdata.loc[
                (forecastdata['RETAILER_ID'] == r) & 
                (forecastdata['PRODUCT_ID'] == k) & 
                (forecastdata['TIME_PERIOD_ID'] == t) & 
                (forecastdata['VERSION_ID'] == version_to_solve),  # Ensure correct version
                'PREDICTION_QUANTITY'
            ].sum()  # Sum in case of duplicate entries

            if t == first_time_period:  # If time period is T37
               mdl.add_constraint(
                    y_vars[r, k, previous_t] + mdl.sum(set(x_vars[j, r, k, t] for j in dc['DC_ID'] if (j, r) in valid_retailer_flows)) ==
                    y_vars[r, k, t] + predicted_demand,
                    ctname=f"RetailerMassBalance_{r}_{k}_t{t}"
                )

            else:  # For all other time periods
                previous_t = list(time_index_mapping.keys())[list(time_index_mapping.values()).index(time_index_mapping[t] - 1)]

                mdl.add_constraint(
                    y_vars[r, k, previous_t] + mdl.sum(set(x_vars[j, r, k, t] for j in dc['DC_ID'] if (j, r) in valid_retailer_flows)) ==
                    y_vars[r, k, t] + predicted_demand,
                    ctname=f"RetailerMassBalance_{r}_{k}_t{t}"
                )


In [ ]:
print(dc_retailer[dc_retailer['VERSION_ID'] == version_to_solve])

### storage capacity constraint

In [31]:
# Convert UNIT_VOLUME to a dictionary for fast lookups
unit_volume_dict = product.set_index('PRODUCT_ID')['UNIT_VOLUME'].to_dict()

# Add Capacity Constraint ---
for j in all_loc_cost_cap['location']:
    for t in filtered_time_period['TIME_PERIOD_ID']:  # Iterate over time periods
        storage_capacity = all_loc_cost_cap.loc[all_loc_cost_cap['location'] == j, 'STORAGE_CAPACITY'].values
        if storage_capacity.size > 0:
            mdl.add_constraint(
                mdl.sum(y_vars[j, k, t] * unit_volume_dict.get(k, 0)  # Fetch from dictionary safely
                        for k in product['PRODUCT_ID'])
                <= storage_capacity[0],
                ctname=f"Storage_Capacity_{j}_t{t}"
            )

locationids = plantids+dcids+retailids

In [ ]:
for c in mdl.iter_constraints():
    print(c)


# This is a test
for j in all_loc_cost_cap['location']:
    for t in filtered_time_period['TIME_PERIOD_ID']:  # Iterate over time periods
        for k in product['PRODUCT_ID']:
            mdl.add_constraint(y_vars[j, k, t] >= 1)

In [33]:
mdl.minimize(
    # 🔹 Storage Cost Minimization
    mdl.sum(storage_cost[j, k] * y_vars[j, k, t] * unit_volume_dict.get(k, 0)
            for (j, k, t) in y_vars.keys()
            if (j, k) in storage_cost and j in locationids and k in productids)
    
    +
    
    # 🔹 Production Cost Minimization
    mdl.sum(prod_cost.get(k, 0) * p_vars[i, k, t]  
            for (i, k, t) in p_vars.keys())
    
)

solution_status = mdl.solve()
# # --- Step 5: Solve Model ---
# solution_status = mdl.solve()

# # --- Step 6: Output Results ---
# if solution_status:
#     print("✅ Optimal solution found!")

#     # Print decision variables with non-zero values
#     for v in mdl.iter_variables():
#         if v.solution_value > 0:
#             print(f"{v.name}: {v.solution_value}")

In [34]:
# --- Step 6: Output Results ---
if solution_status:
    print("✅ Optimal solution found!")

    # Print decision variables with non-zero values
    for v in mdl.iter_variables():
        if v.solution_value > 0:
            print(f"{v.name}: {v.solution_value}")

In [ ]:
# Save all decision variable values, setting negative ones to 0
solution_values = {v.name: max(v.solution_value, 0) for v in mdl.iter_variables()}

# Print only nonzero values for readability
for var_name, value in solution_values.items():
    if value >= 0:
        print(f"{var_name}: {value}")


In [ ]:
import pandas as pd
import re

# Lists to store extracted values
dc_retailer_product_flow_data = []
plant_stock_projection_data = []
retailer_stock_projection_data = []
plant_dc_product_flow_data = []

# Define regex patterns to extract variable names
x_dc_rt_pattern = re.compile(r'X_(DC\d+)_(RT\d+)_(PD\d+)_tT(\d+)')
y_pl_pattern = re.compile(r'Y_(PL\d+)_(PD\d+)_tT(\d+)')
y_rt_pattern = re.compile(r'Y_(RT\d+)_(PD\d+)_tT(\d+)')
x_pl_dc_pattern = re.compile(r'X_(PL\d+)_(DC\d+)_(PD\d+)_tT(\d+)')

# Iterate through solution variables
for v in mdl.iter_variables():
    var_name = v.name
    quantity = v.solution_value  # This might be 0

    # Check for DC-Retailer flow (X_DC_RT_PD_tT)
    x_dc_rt_match = x_dc_rt_pattern.match(var_name)
    if x_dc_rt_match:
        dc_id, retailer_id, product_id, time_period_id = x_dc_rt_match.groups()
        dc_retailer_product_flow_data.append([version_to_solve, dc_id, retailer_id, product_id, f'T{time_period_id}', quantity])

    # Check for Plant inventory (Y_PL_PD_tT)
    y_pl_match = y_pl_pattern.match(var_name)
    if y_pl_match:
        plant_id, product_id, time_period_id = y_pl_match.groups()
        plant_stock_projection_data.append([version_to_solve, plant_id, product_id, f'T{time_period_id}', quantity])

    # ✅ Fix: Include Retailer inventory values even if 0
    y_rt_match = y_rt_pattern.match(var_name)
    if y_rt_match:
        retailer_id, product_id, time_period_id = y_rt_match.groups()
        retailer_stock_projection_data.append([version_to_solve, retailer_id, product_id, f'T{time_period_id}', quantity])

    # Check for Plant-DC flow (X_PL_DC_PD_tT)
    x_pl_dc_match = x_pl_dc_pattern.match(var_name)
    if x_pl_dc_match:
        plant_id, dc_id, product_id, time_period_id = x_pl_dc_match.groups()
        plant_dc_product_flow_data.append([version_to_solve, plant_id, dc_id, product_id, f'T{time_period_id}', quantity])

# Convert lists into DataFrames
dc_retailer_product_flow_df = pd.DataFrame(dc_retailer_product_flow_data, 
                                           columns=["VERSION_ID", "DC_ID", "RETAILER_ID", "PRODUCT_ID", "TIME_PERIOD_ID", "QUANTITY"])

plant_stock_projection_df = pd.DataFrame(plant_stock_projection_data, 
                                         columns=["VERSION_ID", "PLANT_ID", "PRODUCT_ID", "TIME_PERIOD_ID", "INVENTORY"])

retailer_stock_projection_df = pd.DataFrame(retailer_stock_projection_data, 
                                            columns=["VERSION_ID", "RETAILER_ID", "PRODUCT_ID", "TIME_PERIOD_ID", "INVENTORY"])

plant_dc_product_flow_df = pd.DataFrame(plant_dc_product_flow_data, 
                                        columns=["VERSION_ID", "PLANT_ID", "DC_ID", "PRODUCT_ID", "TIME_PERIOD_ID", "QUANTITY"])

# Display the resulting DataFrames
print("DC to Retailer Product Flow DataFrame:")
print(dc_retailer_product_flow_df)

print("\nPlant Stock Projection DataFrame:")
print(plant_stock_projection_df)

print("\nRetailer Stock Projection DataFrame:")
print(retailer_stock_projection_df)  # ✅ Now includes 0 values

print("\nPlant to DC Product Flow DataFrame:")
print(plant_dc_product_flow_df)


In [ ]:
print(filtered_time_period)

In [ ]:
print(plant_product)

In [ ]:
# Ensure filtered_time_period only includes TIME_PERIOD_ID before cross join
time_period_df = filtered_time_period[['TIME_PERIOD_ID']]

# Ensure unique PLANT_ID before mapping
plant_unique = plant.drop_duplicates(subset=['PLANT_ID'])

# PLANT COSTING (Holding)
plant_costing = plant[['VERSION_ID', 'PLANT_ID']].merge(time_period_df, how="cross")
plant_costing['COST_TYPE'] = 'HOLDING'
plant_costing['COST'] = plant_costing['PLANT_ID'].map(plant_unique.set_index('PLANT_ID')['HOLDING_COST'])



# PRODUCTION COST should come from the product table via plant_product mapping
production_costing = plant[['VERSION_ID', 'PLANT_ID']].merge(time_period_df, how="cross")

# Merge with plant_product to get PRODUCT_ID
production_costing = production_costing.merge(
    plant_product[['VERSION_ID', 'PLANT_ID', 'PRODUCT_ID']], 
    on=['VERSION_ID', 'PLANT_ID'], 
    how='inner'
)

# Merge with product to get UNIT_COST
production_costing = production_costing.merge(
    product[['VERSION_ID', 'PRODUCT_ID', 'UNIT_COST']], 
    on=['VERSION_ID', 'PRODUCT_ID'], 
    how='left'
)

# Set cost type as PRODUCTION
production_costing['COST_TYPE'] = 'PRODUCTION'
production_costing.rename(columns={'UNIT_COST': 'COST'}, inplace=True)

# Combine Holding + Production costs
plant_costing = pd.concat([plant_costing, production_costing], ignore_index=True)

### -------- DC COSTING -------- ###
### ---- DC COSTING (Holding + Processing) ---- ###
# Ensure unique DC_ID before mapping
dc_unique = dc.drop_duplicates(subset=['DC_ID'])

# Create DC_COSTING base
dc_costing = dc[['VERSION_ID', 'DC_ID']].merge(time_period_df, how="cross")

# Holding cost at DCs
dc_costing['COST_TYPE'] = 'HOLDING'
dc_costing['COST'] = dc_costing['DC_ID'].map(dc_unique.set_index('DC_ID')['HOLDING_COST'])

# Processing cost at DCs (from dc_product)
processing_costing = dc_product[['VERSION_ID', 'DC_ID', 'PRODUCT_ID']].merge(time_period_df, how="cross")
processing_costing['COST_TYPE'] = 'PROCESSING'
processing_costing['COST'] = 0  # Assuming no direct mapping, you can update this if needed

# Combine Holding + Processing costs for DCs
dc_costing = pd.concat([dc_costing, processing_costing], ignore_index=True)

### ---- RETAILER COSTING (Holding + Missed Sale Penalty) ---- ###
# Ensure unique RETAILER_ID before mapping
retailer_unique = retailer.drop_duplicates(subset=['RETAILER_ID'])

### ---- RETAILER COSTING (Holding + Missed Sale Penalty) ---- ###
retailer_unique = retailer.drop_duplicates(subset=['RETAILER_ID'])
retailer_costing = retailer[['VERSION_ID', 'RETAILER_ID']].merge(time_period_df, how="cross")

# Holding cost
retailer_costing['COST_TYPE'] = 'HOLDING'
retailer_costing['COST'] = retailer_costing['RETAILER_ID'].map(retailer_unique.set_index('RETAILER_ID')['HOLDING_COST'])

# Missed Sale Penalty (Fix: Merge instead of map to avoid duplicates)
penalty_costing = retailer_product[['RETAILER_ID', 'PRODUCT_ID', 'MISSED_SALE_PENALTY_COST']].merge(time_period_df, how="cross")
penalty_costing['COST_TYPE'] = 'MISSED_SALE_PENALTY'
penalty_costing.rename(columns={'MISSED_SALE_PENALTY_COST': 'COST'}, inplace=True)

# Combine Holding + Penalty costs
retailer_costing = pd.concat([retailer_costing, penalty_costing], ignore_index=True)






In [ ]:
print(plant_costing)
print(dc_costing)
print(retailer_costing)



## Plant Product Inventory

In [ ]:
import numpy as np
import pandas as pd

# Ensure TIME_PERIOD_ID is treated as a string before stripping "T"
filtered_time_period['TIME_PERIOD_ID'] = filtered_time_period['TIME_PERIOD_ID'].astype(str)

# Get the first forecasted time period dynamically (smallest time period)
first_forecast_time_period = min(filtered_time_period['TIME_PERIOD_ID'], key=lambda x: int(x.lstrip("T")))  

# Convert "T35" → 35 (ensuring it's always a string first)
first_forecast_time_period_int = int(first_forecast_time_period.lstrip("T"))

# Compute the previous time period correctly
t34_time_period = f"T{first_forecast_time_period_int - 1}"

# Print for debugging
print(f"First Forecasted Time Period: T{first_forecast_time_period_int}, Computed Previous Time Period: {t34_time_period}")

In [ ]:

# Initialize projected stock dictionary for Plants
plant_projected_stock = {}

# Create empty list to store Plant inventory data
plant_inventory_data = []

# Loop over Plants, Products, and Time Periods
for (plant_id, product_id), group in plant_product.groupby(['PLANT_ID', 'PRODUCT_ID']):
    initial_stock = group['INITIAL_STOCK'].values[0]  # Get INITIAL_STOCK
    fill_rate = group['FILL_RATE'].values[0] / 100  # Convert FILL_RATE to decimal (e.g., 98 → 0.98)

    for time_period in sorted(filtered_time_period['TIME_PERIOD_ID']):
        
        # Convert time period to integer
        time_period_int = int(time_period.lstrip("T"))  

        # Set Initial Stock for T24
        if time_period == t34_time_period:  # First value should use T24
            initial_inventory = initial_stock  
        else:
            # Use projected stock from the previous period
            initial_inventory = plant_projected_stock.get((plant_id, product_id, time_period_int - 1), 0)

        # Fetch forecasted demand (assumed as outflow)
        forecasted_demand = forecastdata.loc[
            (forecastdata['PRODUCT_ID'] == product_id) & 
            (forecastdata['TIME_PERIOD_ID'] == f'T{time_period_int}'),
            'PREDICTION_QUANTITY'
        ].sum()

        # Compute Optimized Reorder Point and Safety Stock
        lead_time = 1  # Example lead time
        z = np.abs(np.percentile(forecasted_demand, 100 * (1 - fill_rate)))  # Use dynamic FILL_RATE
        reorder_point = forecasted_demand * lead_time + z
        optimized_ss = max(reorder_point - forecasted_demand * lead_time, 0)

        # Compute optimal order quantity
        order_quantity = np.ceil(forecasted_demand.mean() + z).astype(int)

        # Store the projected stock for the next period
        plant_projected_stock[(plant_id, product_id, time_period_int)] = max(initial_inventory - forecasted_demand, 0)

        # Append to inventory list
        plant_inventory_data.append([
            version_to_solve, plant_id, product_id, time_period_int, 
            0, optimized_ss, reorder_point, order_quantity  # Baseline SS = 0
        ])

# Convert to DataFrame
plant_product_inventory = pd.DataFrame(plant_inventory_data, columns=[
    'VERSION_ID', 'PLANT_ID', 'PRODUCT_ID', 'TIME_PERIOD_ID', 
    'BASELINE_SS', 'OPTIMISED_SS', 'REORDER_POINT', 'REORDER_QUANTITY'
])

# Print or use plant_product_inventory
print(plant_product_inventory)


## DC Product Inventory¶

In [ ]:
import numpy as np
import pandas as pd

# Initialize projected stock dictionary for DCs
dc_projected_stock = {}

# Create empty list to store DC inventory data
dc_inventory_data = []

# Loop over DCs, Products, and Time Periods
for (dc_id, product_id), group in dc_product.groupby(['DC_ID', 'PRODUCT_ID']):
    initial_stock = group['INITIAL_STOCK'].values[0]  # Get INITIAL_STOCK
    fill_rate = group['FILL_RATE'].values[0] / 100  # Convert FILL_RATE to decimal (e.g., 98 → 0.98)

    for time_period in sorted(filtered_time_period['TIME_PERIOD_ID']):
        
        # Convert time period to integer
        time_period_int = int(time_period.lstrip("T"))  

        # Set Initial Stock for T24
        if time_period == t34_time_period:  # First value should use T24
            initial_inventory = initial_stock  
        else:
            # Use projected stock from the previous period
            initial_inventory = dc_projected_stock.get((dc_id, product_id, time_period_int - 1), 0)

        # Fetch forecasted demand (assumed as outflow)
        forecasted_demand = forecastdata.loc[
            (forecastdata['PRODUCT_ID'] == product_id) & 
            (forecastdata['TIME_PERIOD_ID'] == f'T{time_period_int}'),
            'PREDICTION_QUANTITY'
        ].sum()

        # Compute Optimized Reorder Point and Safety Stock
        lead_time = 1  # Example lead time
        z = np.abs(np.percentile(forecasted_demand, 100 * (1 - fill_rate)))  # Use dynamic FILL_RATE
        reorder_point = forecasted_demand * lead_time + z
        optimized_ss = max(reorder_point - forecasted_demand * lead_time, 0)

        # Compute optimal order quantity
        order_quantity = np.ceil(forecasted_demand.mean() + z).astype(int)

        # Store the projected stock for the next period
        dc_projected_stock[(dc_id, product_id, time_period_int)] = max(initial_inventory - forecasted_demand, 0)

        # Append to inventory list
        dc_inventory_data.append([
            version_to_solve, dc_id, product_id, time_period_int, 
            0, optimized_ss, reorder_point, order_quantity  # Baseline SS = 0
        ])

# Convert to DataFrame
dc_product_inventory = pd.DataFrame(dc_inventory_data, columns=[
    'VERSION_ID', 'DC_ID', 'PRODUCT_ID', 'TIME_PERIOD_ID', 
    'BASELINE_SS', 'OPTIMISED_SS', 'REORDER_POINT', 'REORDER_QUANTITY'
])

# Print or use dc_product_inventory
print(dc_product_inventory)


## Retailer Product Inventory

In [ ]:
import numpy as np
import pandas as pd

# Initialize projected stock dictionary for Retailers
retailer_projected_stock = {}

# Create empty list to store Retailer inventory data
retailer_inventory_data = []

# Loop over Retailers, Products, and Time Periods
for (retailer_id, product_id), group in retailer_product.groupby(['RETAILER_ID', 'PRODUCT_ID']):
    initial_stock = group['INITIAL_STOCK'].values[0]  # Get INITIAL_STOCK
    fill_rate = group['FILL_RATE'].values[0] / 100  # Convert FILL_RATE to decimal (e.g., 98 → 0.98)

    for time_period in sorted(filtered_time_period['TIME_PERIOD_ID']):
        # Convert time period to integer
        time_period_int = int(time_period.lstrip("T"))  

        # Set Initial Stock for T24
        if time_period == t34_time_period:  # First value should use T24
            initial_inventory = initial_stock  
        else:
            # Use projected stock from the previous period
            initial_inventory = retailer_projected_stock.get((retailer_id, product_id, time_period_int - 1), 0)

        # Fetch forecasted demand (assumed as outflow)
        forecasted_demand = forecastdata.loc[
            (forecastdata['PRODUCT_ID'] == product_id) & 
            (forecastdata['TIME_PERIOD_ID'] == f'T{time_period_int}'),
            'PREDICTION_QUANTITY'
        ].sum()

        # Compute Optimized Reorder Point and Safety Stock
        lead_time = 1  # Example lead time
        z = np.abs(np.percentile(forecasted_demand, 100 * (1 - fill_rate)))  # Use dynamic FILL_RATE
        reorder_point = forecasted_demand * lead_time + z
        optimized_ss = max(reorder_point - forecasted_demand * lead_time, 0)

        # Compute optimal order quantity
        order_quantity = np.ceil(forecasted_demand.mean() + z).astype(int)

        # Store the projected stock for the next period
        retailer_projected_stock[(retailer_id, product_id, time_period_int)] = max(initial_inventory - forecasted_demand, 0)

        # Append to inventory list
        retailer_inventory_data.append([
            version_to_solve, retailer_id, product_id, time_period_int, 
            0, optimized_ss, reorder_point, order_quantity  # Baseline SS = 0
        ])

# Convert to DataFrame
retailer_product_inventory = pd.DataFrame(retailer_inventory_data, columns=[
    'VERSION_ID', 'RETAILER_ID', 'PRODUCT_ID', 'TIME_PERIOD_ID', 
    'BASELINE_SS', 'OPTIMISED_SS', 'REORDER_POINT', 'REORDER_QUANTITY'
])

# Print or use retailer_product_inventory
print(retailer_product_inventory)


In [ ]:
from io import BytesIO
# Dictionary to store buffers for each DataFrame
buffers = {}

# Create buffers and save each DataFrame
dataframes = {
    "retailer_product_inventory.xlsx": retailer_product_inventory,
    "dc_product_inventory.xlsx": dc_product_inventory,
    "plant_product_inventory.xlsx": plant_product_inventory
    
}

# Save each DataFrame to its own buffer
for filename, df in dataframes.items():
    buffer = BytesIO()
    df.to_excel(buffer, index=False)  # Save DataFrame to buffer
    buffer.seek(0)  # Reset buffer position
    buffers[filename] = buffer.read()  # Store buffer content

# Save all files using Watson Studio Library
for filename, file_data in buffers.items():
    wslib.save_data(filename, data=file_data, overwrite=True)

print("All DataFrames saved successfully!")

In [ ]:
def populate_initial_retailer_stock(retailer_product_df, retailer_stock_projection_df):
    """
    Populates initial stock from retailer_product_df into retailer_stock_projection_df for time period T24.

    Args:
        retailer_product_df (pd.DataFrame): DataFrame containing retailer stock information with INITIAL_STOCK.
        retailer_stock_projection_df (pd.DataFrame): DataFrame where stock projections are stored.

    Returns:
        pd.DataFrame: Updated retailer_stock_projection_df with T24 values populated.
    """
    # Ensure 'TIME_PERIOD_ID' is string (consistent formatting)
    retailer_stock_projection_df["TIME_PERIOD_ID"] = retailer_stock_projection_df["TIME_PERIOD_ID"].astype(str)
    
    # Extract T24 records from retailer_product_df
    t24_records = retailer_product_df[["RETAILER_ID", "PRODUCT_ID", "INITIAL_STOCK"]].copy()
    t24_records["TIME_PERIOD_ID"] = "T24"  # Assign T24 period
    t24_records["VERSION_ID"] = version_to_solve  # Ensure consistent version
    t24_records.rename(columns={"INITIAL_STOCK": "INVENTORY"}, inplace=True)

    # Remove existing T24 records to prevent duplicates
    retailer_stock_projection_df = retailer_stock_projection_df[
        ~((retailer_stock_projection_df["TIME_PERIOD_ID"] == "T24"))
    ]

    # Merge new T24 records
    retailer_stock_projection_df = pd.concat([retailer_stock_projection_df, t24_records], ignore_index=True)

    # Remove any full duplicates
    retailer_stock_projection_df.drop_duplicates(inplace=True)

    return retailer_stock_projection_df


In [ ]:
# Call function to populate initial stock values for T24
retailer_stock_projection_df = populate_initial_retailer_stock(retailer_product, retailer_stock_projection_df)



In [ ]:

# The project token is an authorization token that is used to access project resources like data sources, connections, and used by platform APIs.
from project_lib import Project
project = Project(project_id='48eddbb9-a2f3-47c4-ab2e-a243594f9602', project_access_token='p-2+PMfMcUvlxQrAqNjYBC0Zgw==;K4XqIvVzelwXInLpofOGYQ==:z8ByB3urlLCK7LpBXt4CpsUdUT8A7y3ZB1EUtmpPFaHxCmatIEAzbBHdnFM/xoqRsfF4j8ntO0jiKD/2DCYZlm/hAuyv4guGdg==')
pc = project.project_context

from ibm_watson_studio_lib import access_project_or_space
wslib = access_project_or_space({'token':'p-2+PMfMcUvlxQrAqNjYBC0Zgw==;K4XqIvVzelwXInLpofOGYQ==:z8ByB3urlLCK7LpBXt4CpsUdUT8A7y3ZB1EUtmpPFaHxCmatIEAzbBHdnFM/xoqRsfF4j8ntO0jiKD/2DCYZlm/hAuyv4guGdg=='})


In [ ]:
from io import BytesIO
import pandas as pd


# Dictionary to store buffers for each DataFrame
buffers = {}

# Create buffers and save each DataFrame
dataframes = {
    "dc_retailer_product_flow_df.xlsx": dc_retailer_product_flow_df,
    "plant_stock_projection_df.xlsx": plant_stock_projection_df,
    "retailer_stock_projection_df.xlsx": retailer_stock_projection_df,
    "plant_dc_product_flow_df.xlsx": plant_dc_product_flow_df,
    #"dc_stock_projection_df.xlsx": dc_stock_projection_df,
    "plant_costing.xlsx":plant_costing,
    "dc_costing.xlsx":dc_costing,
    "retailer_costing.xlsx":retailer_costing,
}

# Save each DataFrame to its own buffer
for filename, df in dataframes.items():
    buffer = BytesIO()
    df.to_excel(buffer, index=False)  # Save DataFrame to buffer
    buffer.seek(0)  # Reset buffer position
    buffers[filename] = buffer.read()  # Store buffer content

# Save all files using Watson Studio Library
for filename, file_data in buffers.items():
    wslib.save_data(filename, data=file_data, overwrite=True)

print("All DataFrames saved successfully!")